In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
from anthropic import Anthropic

client = Anthropic()
model = os.getenv("CLAUDE_MODEL")

In [21]:
def add_user_message(messages, text):
	user_message = {"role": "user", "content": text}
	messages.append(user_message)


def add_assistant_message(messages, text):
	assistant_message = {"role": "assistant", "content": text}
	messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=None):
	params = {
		"model" : model,
		"max_tokens" : 1000,
		"messages" : messages,
		"temperature" : temperature,
		"stop_sequences" : stop_sequences,
	}

	if system:
		params["system"] = system
	
	message = client.messages.create(**params)

	return message.content[0].text


Stop Sequences -- ["```"]

When the user sends the initial message, Claude goes "OK, I need to write a full rule and desrcribe it", and describes it by adding external content outside raw json

Then Claude encounters the assistant message (```json), and, because of "role" : "assistant", Claude assumes that it already wrote that out in its response
 - Then Claude goes "Oh, I've already stated the JSON part", and then starts continuing it by writing out the actual JSON.

Once Claude gets to the end of the json, it will want to close off that markdown code it created (the ```). But it encounters that in the stop_sequences, stops the generation entirely, and sends us back a response. 
 - This results in the raw JSON content just by itself


This is extremely good if we want to generate structured data

In [22]:
messages = []

# event bridges are things used in AWS
add_user_message(messages, "Generate a very short event bridge rule as json")
add_assistant_message(messages, "```json")
text = chat(messages, stop_sequences=["```"])

In [23]:
import json

json.loads(text.strip()) # formats in json, removes external "\n". max_tokens must be > 100 so that Claude can generate full JSON

{'Name': 'MyEventRule',
 'EventBusName': 'default',
 'EventPattern': {'source': ['aws.ec2'],
  'detail-type': ['EC2 Instance State-change Notification'],
  'detail': {'state': ['running']}},
 'State': 'ENABLED',
 'Targets': [{'Arn': 'arn:aws:lambda:us-east-1:123456789012:function:MyFunction',
   'Id': '1'}]}